# Module 1: Environment Setup & Model Serving Verification

This notebook verifies the environment configuration, imports config, initializes `LLMClient`, and runs a health check against the vLLM server serving Llama-3-70B-Instruct.

In [ ]:
# Cell 1: Import dependencies and config
import sys
from loguru import logger

# Set up logging
logger.remove()
logger.add(sys.stderr, level="INFO")

from config import get_config
config = get_config()
print("=== App Config Loaded successfully ===")
for key, val in config.model_dump().items():
    print(f"{key}: {val}")

In [ ]:
# Cell 2: Check vLLM Health Endpoint
import httpx
import asyncio

async def check_vllm_health():
    async with httpx.AsyncClient() as client:
        try:
            resp = await client.get(f"{config.vllm_base_url.rstrip('/v1')}/health")
            if resp.status_code == 200:
                print(f"vLLM Server is HEALTHY: {resp.text or 'OK'}")
            else:
                print(f"vLLM Server returned status code {resp.status_code}: {resp.text}")
        except Exception as e:
            print(f"Failed to connect to vLLM server: {e}")

# Since we are in a Jupyter environment, we can run async code using await
await check_vllm_health()

In [ ]:
# Cell 3: Verify LLMClient Functionality
from llm_client import LLMClient

client = LLMClient()
try:
    response = await client.generate(
        system="You are a helpful assistant specialized in finance.",
        user="What is the capital of India? Answer in one word.",
        max_tokens=10
    )
    print(f"LLM response: {response.strip()}")
except Exception as e:
    print(f"LLM generation verification failed: {e}")